# State capacities & demographics vs. the content of K–12 AI guidance

This notebook is for **RQ2**: We want to know whether a state's **demographics**, its **fiscal / infrastructure
capacity**, or its **political leaning** are associated with what its K–12 AI guidance document actually mentions (in terms of the binary coded variables in **RQ1**).

Our *outcomes* are the individual dimensions we hand-coded on the calibration bench (1 = the document addresses it, 0 = it does not). Our *covariates* are the public state datasets we downloaded into `./data/`.

### What we are testing
**Family 1 — demographic responsiveness.** Our hypothesis is that a state with more of a given student group will be more likely to address the dimension that serves that group:

| covariate | we expect it to predict → |
|---|---|
| rural % | rural / underserved |
| English-learner % | English language learners |
| Native (AI/AN) % | tribal / Indigenous |
| disability (IEP) % | accessibility & disability |

**Family 2 — state capacity.** Our hypothesis is that states with more money / infrastructure write more implementation-ready guidance, and that states facing a larger digital divide are more likely to address equity of access. We test **per-pupil spending** against each implementation dimension — PD, funding, upskilling — and
both **broadband** and **free/reduced-lunch share** against the equity / digital-divide dimension.

**Family 3 - Partisan Leaning** we check whether more Democratic legislatures are more likely to address **each** equity / responsible-use dimension.

We use **rank-based** test (Mann-Whitney).

**This notebook** reads a single pre-merged table, `data/RQ2/rq2_state_table.csv` (one row per state, coded outcomes + covariates already joined), and runs every test off it.

## Step 0 — Setup

We load our libraries and read the single pre-merged analysis table, `data/RQ2/rq2_state_table.csv` — one row per coded state, with the RQ1 coded dimensions (1/0) and every covariate already merged in.

The table is [`data/RQ2/rq2_state_table.csv`](../../data/RQ2/rq2_state_table.csv) — click to open it and compare against the values printed in the cells below.

In [55]:
# --- libraries -----------------------------------------------------------
import os, re, json, math, warnings     # os/paths, regex, json, math, warning control
import numpy as np                      # numeric arrays + NaN handling
import pandas as pd                     # tables (our main data structure here)
from scipy import stats                 # our statistical tests (mannwhitneyu)
from scipy.stats import false_discovery_control   # Benjamini-Hochberg FDR (optional robustness)
warnings.filterwarnings("ignore")       # silence version-chatter so the notebook reads cleanly
pd.set_option("display.width", 170); pd.set_option("display.max_columns", 40)  # show wide tables

# --- the single, pre-merged analysis table -------------------------------
# Everything below reads ONE file: rq2_state_table.csv — one row per coded state, with the hand-coded
# RQ1 dimensions (1/0) and all public covariates already merged in. It was built once from the raw
# Census / NCES / Ballotpedia sources + the calibration bench; that build lives in the companion notebook.
HERE = os.getcwd()                                                   # this notebook's folder
DATA = os.path.normpath(os.path.join(HERE, "..", "..", "data", "RQ2"))   # repo data/RQ2
df   = pd.read_csv(os.path.join(DATA, "rq2_state_table.csv"), index_col="state")   # the merged table
print("merged table:", df.shape, "| states:", len(df))
df.head()

merged table: (36, 26) | states: 36


,pages,roadmap,review_cycle,prof_dev,actionable,funding,equity_access,algo_bias,ell,rural_underserved,tribal_indigenous,accessibility_disability,vendor_eval,upskilling,deepfakes,academic_integrity,data_privacy,ai_companion,pct_rural,pct_ell,pct_aian,pct_iep,pct_frl,per_pupil,pct_broadband,dem_share
state,,,,,,,,,,,,,,,,,,,,,,,,,,
Alabama,13.0,0,0,1,1,0,1,1,0,0,0,0,1,1,0,1,1,0,42.262760,4.672754,0.833108,13.306424,60.161118,13626.861805,89.4,26.4
Alaska,15.0,0,0,1,1,1,1,1,0,1,0,0,1,1,0,1,1,0,35.100513,10.786954,21.617466,14.816061,39.478639,21003.794488,93.1,38.3
California,57.0,1,0,1,1,1,1,1,1,1,0,1,1,1,1,1,1,1,5.763367,18.920367,0.448561,13.578833,58.240838,20790.840051,94.2,75.0
Colorado,55.0,1,1,1,1,1,1,1,1,1,0,1,1,1,1,1,1,0,13.973293,10.436897,0.623169,13.008700,42.366906,15907.945284,94.5,66.0
Connecticut,13.0,0,0,1,1,0,1,1,0,0,0,0,1,0,1,1,1,1,13.749271,9.365412,0.250115,17.260517,41.924430,26315.571054,92.6,67.9


## Step 1 — Our coded outcomes

The coded dimensions were hand-coded on the calibration bench (1 = the document addresses it, 0 = it does not).

In [56]:
# Map each coding-question index to a short dimension name.
LABELS = {8:"roadmap",9:"review_cycle",10:"prof_dev",12:"actionable",18:"funding",22:"equity_access",
 23:"algo_bias",24:"ell",27:"rural_underserved",28:"tribal_indigenous",29:"accessibility_disability",25:"vendor_eval",33:"upskilling",
 21:"deepfakes",30:"academic_integrity",31:"data_privacy",32:"ai_companion"}
ACTION_DIMS = [8,9,10,12,18,25]       # implementation-readiness dimensions (we test each one separately)
EQUITY_DIMS = [22,23,24,27,28,29]     # equity / responsible-use dimensions (we test each one separately)

O = df[[LABELS[q] for q in LABELS]]                               # our coded outcomes = the dimension columns of the merged table
print("coded jurisdictions:", len(O))                            # sanity check: we expect 36
O.head()

coded jurisdictions: 36


,roadmap,review_cycle,prof_dev,actionable,funding,equity_access,algo_bias,ell,rural_underserved,tribal_indigenous,accessibility_disability,vendor_eval,upskilling,deepfakes,academic_integrity,data_privacy,ai_companion
state,,,,,,,,,,,,,,,,,
Alabama,0,0,1,1,0,1,1,0,0,0,0,1,1,0,1,1,0
Alaska,0,0,1,1,1,1,1,0,1,0,0,1,1,0,1,1,0
California,1,0,1,1,1,1,1,1,1,0,1,1,1,1,1,1,1
Colorado,1,1,1,1,1,1,1,1,1,0,1,1,1,1,1,1,0
Connecticut,0,0,1,1,0,1,1,0,0,0,0,1,0,1,1,1,1


In [57]:
O

,roadmap,review_cycle,prof_dev,actionable,funding,equity_access,algo_bias,ell,rural_underserved,tribal_indigenous,accessibility_disability,vendor_eval,upskilling,deepfakes,academic_integrity,data_privacy,ai_companion
state,,,,,,,,,,,,,,,,,
Alabama,0,0,1,1,0,1,1,0,0,0,0,1,1,0,1,1,0
Alaska,0,0,1,1,1,1,1,0,1,0,0,1,1,0,1,1,0
California,1,0,1,1,1,1,1,1,1,0,1,1,1,1,1,1,1
Colorado,1,1,1,1,1,1,1,1,1,0,1,1,1,1,1,1,0
Connecticut,0,0,1,1,0,1,1,0,0,0,0,1,0,1,1,1,1
Delaware,1,1,1,1,0,1,1,0,0,0,1,1,1,1,1,1,0
Georgia,1,1,1,1,1,1,1,0,0,0,1,1,0,0,1,1,0
Hawaii,0,1,1,1,0,1,1,0,0,0,0,0,0,0,1,1,0
Illinois,1,0,1,1,1,1,1,1,1,0,1,1,1,1,1,1,1


**Quick look.** Check percentage for each dimension as a spot check and hopefully it matches with the sheet: https://docs.google.com/spreadsheets/d/1igqtFauxSOcQmlrMCPFfKHXN9fq7OBqdppHadrptPxQ/edit?usp=sharing (calibration_with_claude)

In [58]:
# mean of a 0/1 column = share of states that address it; sort high-to-low to see ceilings and rare dimensions
O[[LABELS[q] for q in LABELS]].mean().sort_values(ascending=False).round(3).to_frame("share_yes")

,share_yes
academic_integrity,0.972
algo_bias,0.972
data_privacy,0.972
prof_dev,0.917
equity_access,0.861
upskilling,0.861
vendor_eval,0.833
actionable,0.778
accessibility_disability,0.750
roadmap,0.556


## Step 2 — The covariates

Each covariate is already a column in the merged table (built from the public Census / NCES / Ballotpedia sources). 

**Rural %** — 2020 Census DHC, Table P2. We compute the rural share as rural population ÷ total population, because we want the share of *people* in rural areas, not a count.  
📎 Source: [data.census.gov — DECENNIALDHC2020.P2](https://data.census.gov/table/DECENNIALDHC2020.P2)

In [59]:
s = df["pct_rural"].dropna()                                      # rural share (%), from the merged table
print("n =", s.size, "| most rural:"); s.sort_values(ascending=False).head().round(1)

n = 36 | most rural:


state
Maine            61.4
West Virginia    55.4
Mississippi      53.7
Montana          46.6
Alabama          42.3
Name: pct_rural, dtype: float64

In [60]:
print("n =", df["pct_rural"].notna().sum(), "| most rural:"); df["pct_rural"].round(1)

n = 36 | most rural:


state
Alabama           42.3
Alaska            35.1
California         5.8
Colorado          14.0
Connecticut       13.7
Delaware          17.4
Georgia           25.9
Hawaii            13.9
Illinois          13.1
Indiana           28.8
Kentucky          41.3
Louisiana         28.5
Maine             61.4
Maryland          14.4
Massachusetts      8.7
Michigan          26.5
Minnesota         28.1
Mississippi       53.7
Missouri          30.5
Montana           46.6
Nevada             5.9
New Jersey         6.2
New Mexico        25.5
North Carolina    33.3
North Dakota      39.0
Ohio              23.7
Oklahoma          35.4
Oregon            19.5
Puerto Rico        8.1
Rhode Island       8.9
Utah              10.2
Virginia          24.4
Washington        16.6
West Virginia     55.4
Wisconsin         32.9
Wyoming           38.0
Name: pct_rural, dtype: float64

**English-learner %** — NCES Digest 2023 Table 204.20. EL students as a percent of enrollment (Fall 2021). We located the percent column by inspecting the sheet: it is column index 22.  
📎 Source: [NCES Digest Table 204.20](https://nces.ed.gov/programs/digest/d23/tables/dt23_204.20.asp) · [direct .xlsx](https://nces.ed.gov/programs/digest/d23/tables/xls/tabn204.20.xlsx)

In [62]:
s = df["pct_ell"].dropna()                                        # English-learner share (%)
print("n =", s.size, "| highest ELL:"); s.sort_values(ascending=False).head().round(1)

n = 36 | highest ELL:


state
California      18.9
New Mexico      18.8
Nevada          13.8
Illinois        12.8
Rhode Island    12.5
Name: pct_ell, dtype: float64

In [63]:
print("n =", df["pct_ell"].notna().sum(), "| highest ELL:"); df["pct_ell"].round(1)

n = 36 | highest ELL:


state
Alabama            4.7
Alaska            10.8
California        18.9
Colorado          10.4
Connecticut        9.4
Delaware          11.5
Georgia            7.7
Hawaii            10.0
Illinois          12.8
Indiana            7.0
Kentucky           5.4
Louisiana          4.9
Maine              3.1
Maryland          11.2
Massachusetts     10.5
Michigan           6.4
Minnesota          8.8
Mississippi        3.1
Missouri           3.8
Montana            2.4
Nevada            13.8
New Jersey         8.2
New Mexico        18.8
North Carolina     8.0
North Dakota       3.3
Ohio               3.8
Oklahoma           9.3
Oregon             9.5
Puerto Rico        0.3
Rhode Island      12.5
Utah               8.3
Virginia           9.4
Washington        11.4
West Virginia      0.8
Wisconsin          5.9
Wyoming            2.7
Name: pct_ell, dtype: float64

**Native (AI/AN) %** — NCES Digest 2023 Table 203.70 (race/ethnicity distribution, Fall 2022). Already a percent; the American Indian/Alaska Native column is index 2.  
📎 Source: [NCES Digest Table 203.70](https://nces.ed.gov/programs/digest/d23/tables/dt23_203.70.asp) · [direct .xlsx](https://nces.ed.gov/programs/digest/d23/tables/xls/tabn203.70.xlsx)

In [64]:
s = df["pct_aian"].dropna()                                       # American Indian / Alaska Native share (%)
print("n =", s.size, "| highest Native share:"); s.sort_values(ascending=False).head().round(1)

n = 36 | highest Native share:


state
Alaska          21.6
Oklahoma        11.2
Montana         10.4
New Mexico      10.3
North Dakota     8.2
Name: pct_aian, dtype: float64

In [65]:
print("n =", df["pct_aian"].notna().sum(), "| highest Native share:"); df["pct_aian"].round(1)

n = 36 | highest Native share:


state
Alabama            0.8
Alaska            21.6
California         0.4
Colorado           0.6
Connecticut        0.3
Delaware           0.4
Georgia            0.2
Hawaii             0.1
Illinois           0.2
Indiana            0.2
Kentucky           0.1
Louisiana          0.6
Maine              0.8
Maryland           0.3
Massachusetts      0.2
Michigan           0.6
Minnesota          1.7
Mississippi        0.2
Missouri           0.4
Montana           10.4
Nevada             0.8
New Jersey         0.2
New Mexico        10.3
North Carolina     1.1
North Dakota       8.2
Ohio               0.1
Oklahoma          11.2
Oregon             1.2
Puerto Rico        0.0
Rhode Island       0.7
Utah               1.0
Virginia           0.3
Washington         1.1
West Virginia      0.1
Wisconsin          1.0
Wyoming            3.1
Name: pct_aian, dtype: float64

**Disability (IEP) %** — NCES Digest 2023 Table 204.70. Students served under IDEA Part B as a percent of enrollment (2022–23); that percent is in column index 14.  
📎 Source: [NCES Digest Table 204.70](https://nces.ed.gov/programs/digest/d23/tables/dt23_204.70.asp) · [direct .xlsx](https://nces.ed.gov/programs/digest/d23/tables/xls/tabn204.70.xlsx)

In [66]:
s = df["pct_iep"].dropna()                                        # students served under IDEA (%)
print("n =", s.size, "| highest IEP share:"); s.sort_values(ascending=False).head().round(1)

n = 36 | highest IEP share:


state
Puerto Rico      36.8
Maine            20.6
Massachusetts    19.7
Delaware         19.4
West Virginia    18.7
Name: pct_iep, dtype: float64

In [67]:
print("n =", df["pct_iep"].notna().sum(), "| highest IEP share:"); df["pct_iep"].round(1)

n = 36 | highest IEP share:


state
Alabama           13.3
Alaska            14.8
California        13.6
Colorado          13.0
Connecticut       17.3
Delaware          19.4
Georgia           13.2
Hawaii            11.7
Illinois          15.9
Indiana           18.2
Kentucky          16.6
Louisiana         13.5
Maine             20.6
Maryland          12.9
Massachusetts     19.7
Michigan          13.9
Minnesota         17.5
Mississippi       15.7
Missouri          14.3
Montana           14.0
Nevada            13.2
New Jersey        17.6
New Mexico        17.8
North Carolina    13.1
North Dakota      14.9
Ohio              16.6
Oklahoma          17.2
Oregon            15.2
Puerto Rico       36.8
Rhode Island      17.9
Utah              13.3
Virginia          14.1
Washington        13.9
West Virginia     18.7
Wisconsin         15.2
Wyoming           17.7
Name: pct_iep, dtype: float64

**Free/reduced-lunch %** — NCES Digest 2023 Table 204.10 (eligible, 2022–23 = column 28). We keep this but treat it cautiously: pandemic universal-meal waivers inflate/depress some states in these years, so it is a noisy poverty proxy.  
📎 Source: [NCES Digest Table 204.10](https://nces.ed.gov/programs/digest/d23/tables/dt23_204.10.asp) · [direct .xlsx](https://nces.ed.gov/programs/digest/d23/tables/xls/tabn204.10.xlsx)

In [68]:
s = df["pct_frl"].dropna()                                        # free/reduced-price lunch eligibility (%)
print("n =", s.size, "| highest FRL share:"); s.sort_values(ascending=False).head().round(1)

n = 35 | highest FRL share:


state
Mississippi       99.7
Nevada            81.5
New Mexico        75.2
North Carolina    62.5
Louisiana         62.4
Name: pct_frl, dtype: float64

**Per-pupil spending** — Census F-33 FY2024, sheet `8`, total current dollars per pupil (column index 2). We use the F-33 survey as the machine-readable stand-in for the NCES 2023–24 First Look (same federal data); 50 states + DC, no Puerto Rico.  
📎 Source: [Census school-finance tables](https://www.census.gov/programs-surveys/school-finances/data/tables.html) · [direct .xlsx](https://www2.census.gov/programs-surveys/school-finances/tables/2024/secondary-education-finance/elsec24_sumtables.xlsx)

In [69]:
s = df["per_pupil"].dropna()                                      # total current spending per pupil ($)
print("n =", s.size, "| highest spenders:"); s.sort_values(ascending=False).head().round(0)

n = 35 | highest spenders:


state
New Jersey       27234.0
Connecticut      26316.0
Massachusetts    24429.0
Hawaii           23878.0
Delaware         22201.0
Name: per_pupil, dtype: float64

In [70]:
print("n =", df["per_pupil"].notna().sum(), "| highest spenders:"); df["per_pupil"].head().round(0)

n = 35 | highest spenders:


state
Alabama        13627.0
Alaska         21004.0
California     20791.0
Colorado       15908.0
Connecticut    26316.0
Name: per_pupil, dtype: float64

**Broadband %** — ACS 1-Year 2023 Table S2801, households with any broadband subscription (variable `S2801_C02_014E`). We include this as an infrastructure / digital-divide capacity measure.  
📎 Source: [data.census.gov — ACSST1Y2023.S2801](https://data.census.gov/table/ACSST1Y2023.S2801)

In [71]:
s = df["pct_broadband"].dropna()                                  # households with any broadband (%)
print("n =", s.size, "| lowest broadband:"); s.sort_values().head().round(1)

n = 36 | lowest broadband:


state
Puerto Rico      85.5
Mississippi      86.5
West Virginia    86.8
Louisiana        88.0
New Mexico       88.6
Name: pct_broadband, dtype: float64

In [72]:
print("n =", df["pct_broadband"].notna().sum(), "| lowest broadband:"); df["pct_broadband"].round(1)

n = 36 | lowest broadband:


state
Alabama           89.4
Alaska            93.1
California        94.2
Colorado          94.5
Connecticut       92.6
Delaware          92.8
Georgia           92.4
Hawaii            92.6
Illinois          91.6
Indiana           90.8
Kentucky          90.1
Louisiana         88.0
Maine             91.8
Maryland          92.9
Massachusetts     93.9
Michigan          91.6
Minnesota         92.2
Mississippi       86.5
Missouri          90.7
Montana           90.9
Nevada            93.9
New Jersey        94.0
New Mexico        88.6
North Carolina    91.5
North Dakota      89.0
Ohio              91.2
Oklahoma          90.2
Oregon            93.3
Puerto Rico       85.5
Rhode Island      92.9
Utah              94.1
Virginia          92.1
Washington        94.1
West Virginia     86.8
Wisconsin         91.6
Wyoming           90.5
Name: pct_broadband, dtype: float64

**Partisan lean** — Ballotpedia, partisan composition of state legislatures. We use the **Democratic share of all
legislative seats** (both chambers combined), taken directly from Ballotpedia’s overall-balance table — no averaging.
It is a continuous 0–100 measure (not a Dem/Rep label) so we keep more information.  
📎 Source: [Ballotpedia — Partisan composition of state legislatures](https://ballotpedia.org/Partisan_composition_of_state_legislatures) (page as of Oct 16, 2026)

In [74]:
s = df["dem_share"].dropna()                                      # Democratic share of all legislative seats (%)
print("n =", s.size, "| most Democratic legislatures:"); s.sort_values(ascending=False).head().round(1)

n = 35 | most Democratic legislatures:


state
Rhode Island     85.8
Hawaii           84.2
Massachusetts    83.5
California       75.0
Maryland         72.3
Name: dem_share, dtype: float64

In [75]:
df["dem_share"].round(1)

state
Alabama           26.4
Alaska            38.3
California        75.0
Colorado          66.0
Connecticut       67.9
Delaware          67.7
Georgia           43.6
Hawaii            84.2
Illinois          66.7
Indiana           26.7
Kentucky          19.6
Louisiana         29.2
Maine             51.6
Maryland          72.3
Massachusetts     83.5
Michigan          48.0
Minnesota         50.2
Mississippi       32.2
Missouri          31.5
Montana           40.0
Nevada            63.5
New Jersey        64.2
New Mexico        62.5
North Carolina    40.6
North Dakota      11.3
Ohio              32.6
Oklahoma          17.4
Oregon            58.9
Puerto Rico        NaN
Rhode Island      85.8
Utah              19.2
Virginia          51.4
Washington        59.9
West Virginia      8.2
Wisconsin         45.5
Wyoming            8.6
Name: dem_share, dtype: float64

## Step 3 — The merged analysis table

All covariates are already left-joined onto our coded outcomes in `rq2_state_table.csv`. Where a source does not list a jurisdiction (e.g. no Puerto Rico in the spending survey) the cell is missing, and every test below simply drops missing rows for that pair — we never impute. Below is a quick sanity-check of coverage.

In [76]:
# The merge was done in advance (see the build notebook); here we only sanity-check coverage on the loaded table.
COV_COLS = ["pct_rural","pct_ell","pct_aian","pct_iep","pct_frl","per_pupil","pct_broadband","dem_share"]
print("merged table:", df.shape)
print("\nnon-null covariate counts (out of %d):" % len(df))
print(df[COV_COLS].notna().sum())
df[["pages","rural_underserved","tribal_indigenous"] + COV_COLS].round(1).head(10)

merged table: (36, 26)

non-null covariate counts (out of 36):
pct_rural        36
pct_ell          36
pct_aian         36
pct_iep          36
pct_frl          35
per_pupil        35
pct_broadband    36
dem_share        35
dtype: int64


,pages,rural_underserved,tribal_indigenous,pct_rural,pct_ell,pct_aian,pct_iep,pct_frl,per_pupil,pct_broadband,dem_share
state,,,,,,,,,,,
Alabama,13.0,0,0,42.3,4.7,0.8,13.3,60.2,13626.9,89.4,26.4
Alaska,15.0,1,0,35.1,10.8,21.6,14.8,39.5,21003.8,93.1,38.3
California,57.0,1,0,5.8,18.9,0.4,13.6,58.2,20790.8,94.2,75.0
Colorado,55.0,1,0,14.0,10.4,0.6,13.0,42.4,15907.9,94.5,66.0
Connecticut,13.0,0,0,13.7,9.4,0.3,17.3,41.9,26315.6,92.6,67.9
Delaware,28.0,0,0,17.4,11.5,0.4,19.4,25.2,22201.2,92.8,67.7
Georgia,24.0,0,0,25.9,7.7,0.2,13.2,59.6,15832.6,92.4,43.6
Hawaii,2.0,0,0,13.9,10.0,0.1,11.7,46.1,23878.2,92.6,84.2
Illinois,408.0,1,0,13.1,12.8,0.2,15.9,53.7,21776.4,91.6,66.7


## Step 4 — Family 1: demographic responsiveness

**Our question.** Does a higher share of a student group make a state more likely to address that group's dimension?

**The test we use and why.** Our outcomes are single yes/no dimensions and our covariates are continuous shares, so for each
pair we compare the demographic share in states that **do** address the dimension (Yes) against states that **do not** (No),
using the **Mann-Whitney U** test. We chose it deliberately:
- Our groups are small and unequal (as few as 5 Yes-states), so a *t*-test's normality assumption is unsafe.
- The shares are skewed and bounded, with big outliers (Alaska's Native share), and Mann-Whitney compares the whole rank
  ordering, so it is robust to those outliers and to non-normal shapes.
- It answers exactly our question — "is the share systematically higher on one side?" — without assuming a functional form.

We get medians for each side, the **rank-biserial** correlation as a signed effect size (+ = Yes-states higher), and the
two-sided p. **These are four hypotheses we specified a priori, each on its own substantive grounds (a larger group → the
dimension that serves it), and we report all four regardless of outcome — so we treat them as four *separate* confirmatory
tests and base each conclusion on its own raw p-value, not on a family-wise multiple-comparison correction** (correction is
for multiplicity — fishing across many tests or propping up a single combined claim — which is not what four pre-specified,
individually-reported hypotheses are; cf. Rothman 1990; Perneger 1998). For transparency we still print a Benjamini–Hochberg
FDR column (`p_fdr`) as an optional robustness check, but it is not our decision rule. `mwu()` runs one test; `mwu_pairs()`
runs a list of them (and appends the optional FDR column).

In [77]:
def mwu(cov, dim):
    """Mann-Whitney U comparing covariate `cov` between Yes- and No-states on binary outcome `dim`."""
    s  = df[[cov,dim]].dropna()                                   # drop rows missing either value
    g1 = s.loc[s[dim]==1, cov]                                    # covariate values where the dimension IS addressed
    g0 = s.loc[s[dim]==0, cov]                                    # covariate values where it is NOT
    if len(g1)<3 or len(g0)<3: return None                       # skip if either group is too small to compare
    U,p = stats.mannwhitneyu(g1, g0, alternative="two-sided")    # U statistic (for g1) + two-sided p-value
    rb  = 2*U/(len(g1)*len(g0)) - 1                              # rank-biserial effect: + => Yes-states rank higher
    return dict(n_yes=len(g1), n_no=len(g0),                      # group sizes
                med_yes=round(g1.median(),2), med_no=round(g0.median(),2),  # medians for context
                rank_biserial=round(rb,2), p=round(p,3))          # effect size + p

def mwu_pairs(pairs):
    """Run mwu() over (covariate, dimension) pairs; raw p is primary. p_fdr is an OPTIONAL robustness column only."""
    rows = [dict(covariate=c, dimension=d, **mwu(c,d)) for c,d in pairs if mwu(c,d)]  # one row per testable pair
    T = pd.DataFrame(rows)                                        # assemble the results table
    if len(T): T["p_fdr"] = false_discovery_control(T["p"], method="bh").round(3)     # BH-FDR, shown only as a cross-check
    return T

# Our FOUR pre-specified demographic-group hypotheses. Each was formed a priori on its own substantive grounds, and we
# report all four, so we treat them as four SEPARATE confirmatory tests: inference is on the raw two-sided p (the p_fdr
# column is printed only as an optional robustness check, not our decision rule).
MATCH = [("pct_rural","rural_underserved"), ("pct_ell","ell"), ("pct_aian","tribal_indigenous"),
         ("pct_iep","accessibility_disability")]
F1 = mwu_pairs(MATCH)
print("Family 1 — four pre-specified demographic-group hypotheses (inference on raw p; p_fdr = optional robustness):")
F1


Family 1 — four pre-specified demographic-group hypotheses (inference on raw p; p_fdr = optional robustness):


,covariate,dimension,n_yes,n_no,med_yes,med_no,rank_biserial,p,p_fdr
0,pct_rural,rural_underserved,15,21,16.63,28.12,-0.27,0.178,0.356
1,pct_ell,ell,18,18,9.42,7.33,0.19,0.351,0.468
2,pct_aian,tribal_indigenous,5,31,10.27,0.41,0.83,0.001,0.004
3,pct_iep,accessibility_disability,27,9,15.18,16.63,-0.05,0.826,0.826


**How tp read this.** A small `p` with a **positive** `rank_biserial` is the pattern we predicted (states that address the dimension really do have a higher share of that group). The `p_fdr` column is only an optional robustness cross-check (would the result still stand if we treated the four as one family?) — it is not the decision rule.

In [78]:
# Native enrollment % next to our tribal/Indigenous coding, one row per state, sorted by Native share.
tribal_check = (df[["pct_aian","tribal_indigenous"]].dropna()                 # Native % + tribal coding, drop missing
    .assign(coded_tribal=lambda t: t["tribal_indigenous"].map({1:"Yes", 0:"No"}))  # 1/0 -> readable Yes/No
    .rename(columns={"pct_aian":"native_pct"})                               # clearer column name
    .drop(columns="tribal_indigenous")                                       # keep just the % and the Yes/No label
    .sort_values("native_pct", ascending=False).round(2))                    # highest Native share first
print("states addressing tribal/Indigenous:", int((tribal_check.coded_tribal=="Yes").sum()), "of", len(tribal_check))
tribal_check.sort_values(by='native_pct',ascending=False)

states addressing tribal/Indigenous: 5 of 36


,native_pct,coded_tribal
state,,
Alaska,21.62,No
Oklahoma,11.21,Yes
Montana,10.43,Yes
New Mexico,10.27,Yes
North Dakota,8.19,No
Wyoming,3.12,No
Minnesota,1.72,No
Oregon,1.19,Yes
Washington,1.12,No


In [79]:
def check_pair(cov, dim):
    """Per-state table: the demographic % next to our Yes/No coding of `dim`, sorted high-to-low on the %."""
    t = (df[[cov, dim]].dropna()                                 # the two columns, complete cases only
         .assign(**{"coded_"+dim: lambda x: x[dim].map({1:"Yes", 0:"No"})})  # 1/0 -> readable Yes/No label
         .drop(columns=dim)                                      # keep the % and the Yes/No label
         .sort_values(cov, ascending=False).round(2))           # highest share first
    print("%s: %d of %d states coded Yes" % (dim, int((t["coded_"+dim]=="Yes").sum()), len(t)))  # quick count
    return t

**Rural % vs rural/underserved coding**.

In [80]:
check_pair("pct_rural", "rural_underserved").sort_values(by='pct_rural',ascending=False)

rural_underserved: 15 of 36 states coded Yes


,pct_rural,coded_rural_underserved
state,,
Maine,61.37,No
West Virginia,55.35,No
Mississippi,53.71,No
Montana,46.58,Yes
Alabama,42.26,No
Kentucky,41.30,No
North Dakota,39.03,Yes
Wyoming,37.98,No
Oklahoma,35.38,Yes


**English-learner % vs ELL coding.**

In [81]:
check_pair("pct_ell", "ell")

ell: 18 of 36 states coded Yes


,pct_ell,coded_ell
state,,
California,18.92,Yes
New Mexico,18.80,No
Nevada,13.77,Yes
Illinois,12.82,Yes
Rhode Island,12.48,Yes
Delaware,11.54,No
Washington,11.44,Yes
Maryland,11.18,Yes
Alaska,10.79,No


**Disability (IEP) % vs accessibility/disability coding.**

In [82]:
check_pair("pct_iep", "accessibility_disability")

accessibility_disability: 27 of 36 states coded Yes


,pct_iep,coded_accessibility_disability
state,,
Puerto Rico,36.82,Yes
Maine,20.61,Yes
Massachusetts,19.72,Yes
Delaware,19.44,Yes
West Virginia,18.70,Yes
Indiana,18.15,No
Rhode Island,17.93,Yes
New Mexico,17.84,Yes
Wyoming,17.68,No


**Free/reduced-lunch % vs equity/digital-divide coding** (remember the pandemic meal-waiver caveat on FRL).

In [83]:
check_pair("pct_frl", "equity_access")

equity_access: 30 of 35 states coded Yes


,pct_frl,coded_equity_access
state,,
Mississippi,99.70,Yes
Nevada,81.52,Yes
New Mexico,75.24,Yes
North Carolina,62.54,Yes
Louisiana,62.40,No
Oklahoma,60.28,Yes
Alabama,60.16,Yes
Georgia,59.58,Yes
California,58.24,Yes


## Step 5 — Family 2: state capacity

**Our question.** Do states with more fiscal capacity or infrastructure produce more implementation-ready guidance, and are states facing a larger digital divide more likely to address equity of access?


In [84]:
# Family 2 pairs. per-pupil spending -> each implementation dimension; broadband & FRL -> equity/digital divide.
PUPIL_DIMS = [10, 18, 33]             # prof_dev, funding, upskilling
cap_pairs = ([("per_pupil", LABELS[q]) for q in PUPIL_DIMS]
             + [("pct_broadband", "equity_access"), ("pct_frl", "equity_access")])
F2 = mwu_pairs(cap_pairs)                                         # raw p is the basis (separate pre-specified tests)
# We report only the two medians and the p-value for each pair:
F2[["covariate", "dimension", "n_yes", "n_no", "med_yes", "med_no", "p"]]


,covariate,dimension,n_yes,n_no,med_yes,med_no,p
0,per_pupil,prof_dev,32,3,17180.56,13622.21,0.481
1,per_pupil,funding,16,19,17785.26,17102.46,0.608
2,per_pupil,upskilling,30,5,17100.22,20520.93,0.086
3,pct_broadband,equity_access,31,5,91.80,91.20,0.522
4,pct_frl,equity_access,30,5,48.59,54.58,0.873


*Read it as:* a positive `rank_biserial` with a small `p` would mean higher-spending states are more likely to address
that implementation dimension. `p_fdr` is the column that accounts for testing several dimensions at once.

## Step 6 — Family 3: partisan divide

**Our question.** Are more Democratic legislatures more likely to address each **equity** and **responsible-use** dimension?

**First, the per-state table we run these tests on.** we pull each state's Democratic seat share next to the individual
equity dimensions (1 = addressed), sorted from most to least Democratic. This is the raw layout the tests operate on.

In [85]:
EQUITY_F3 = [22, 24, 27, 28, 29]      # equity: equity_access, ELL, rural, tribal, disability
RESP_F3   = [23, 30, 31, 21, 32]      # responsible use: algo_bias, academic_integrity, data_privacy, deepfakes, ai_companion
fam3_dims = [LABELS[q] for q in EQUITY_F3 + RESP_F3]              # the ten equity + responsible-use dimensions
step6_df = (df[["dem_share"] + fam3_dims]                         # partisanship + the binary dimensions
            .dropna(subset=["dem_share"])                         # drop jurisdictions with no legislature share (PR)
            .sort_values("dem_share", ascending=False).round(2))  # most Democratic legislature at the top
print("states with a partisan score:", len(step6_df))
step6_df

states with a partisan score: 35


,dem_share,equity_access,ell,rural_underserved,tribal_indigenous,accessibility_disability,algo_bias,academic_integrity,data_privacy,deepfakes,ai_companion
state,,,,,,,,,,,
Rhode Island,85.8,1,1,1,0,1,1,1,1,1,0
Hawaii,84.2,1,0,0,0,0,1,1,1,0,0
Massachusetts,83.5,1,1,1,0,1,1,1,1,1,0
California,75.0,1,1,1,0,1,1,1,1,1,1
Maryland,72.3,1,1,1,0,1,1,1,1,0,0
Connecticut,67.9,1,0,0,0,0,1,1,1,1,1
Delaware,67.7,1,0,0,0,1,1,1,1,1,0
Illinois,66.7,1,1,1,0,1,1,1,1,1,1
Colorado,66.0,1,1,1,0,1,1,1,1,1,0


In [86]:
# Mann-Whitney per dimension: is Democratic seat-share higher in states that address it? (raw p; separate tests)
PART = mwu_pairs([("dem_share", d) for d in fam3_dims])           # one test per equity / responsible-use dimension
PART["yes_share"] = [round(df[d].mean(), 2) for d in PART["dimension"]]  # overall coverage, to spot ceiling effects

# Dimensions too near-universal to test (a Yes- or No-group < 3) are NOT nulls -- they are bipartisan consensus.
tested = set(PART["dimension"])
consensus = [(d, int(df[d].sum()), int((df[d] == 0).sum())) for d in fam3_dims if d not in tested]
if consensus:
    print("Bipartisan consensus -- addressed by almost all states, too little variation to test:")
    for d, y, n in consensus:
        print("   %-20s Yes %2d / No %2d  (%.0f%% coverage)" % (d, y, n, 100 * y / (y + n)))
    print()

# We report only the two medians and the p-value (yes_share is context for near-universal dimensions):
PART[["covariate", "dimension", "n_yes", "n_no", "med_yes", "med_no", "p", "yes_share"]]

Bipartisan consensus -- addressed by almost all states, too little variation to test:
   algo_bias            Yes 35 / No  1  (97% coverage)
   academic_integrity   Yes 35 / No  1  (97% coverage)
   data_privacy         Yes 35 / No  1  (97% coverage)



,covariate,dimension,n_yes,n_no,med_yes,med_no,p,yes_share
0,dem_share,equity_access,30,5,49.10,32.6,0.448,0.86
1,dem_share,ell,18,17,49.75,48.0,0.934,0.50
2,dem_share,rural_underserved,15,20,62.50,38.1,0.051,0.42
3,dem_share,tribal_indigenous,5,30,45.50,49.1,0.697,0.14
4,dem_share,accessibility_disability,26,9,49.10,38.3,0.559,0.75
5,dem_share,deepfakes,18,17,59.40,45.5,0.478,0.50
6,dem_share,ai_companion,8,27,49.75,48.0,0.954,0.22
